# Geração dos tipos de contradição citados no LegalWiz

Neste notebook modificamos os prompts de geração das contradição para enfatizar um certo tipo de contradição. Também é fornecido vários exemplos para guiar o modelo nessa geração. Além disso foi realizado testes modificando a temperatura do modelo.

In [41]:
# =========================
# LLM STRATEGY / ADAPTER
# =========================

import os
from abc import ABC, abstractmethod
from dotenv import load_dotenv
from typing import Optional
import json

import sys
sys.path.append("../")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
LLM_TEMPERATURE = 0.2


class LLMStrategy(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        pass


class OpenAIAdapter(LLMStrategy):
    def __init__(self, api_key: Optional[str], model: str, temperature: float = 0.2):
        if not api_key:
            raise ValueError("OPENAI_API_KEY não encontrado.")
        from openai import OpenAI
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.temperature = temperature

    def generate(self, prompt: str) -> str:
        print("Temperatura do modelo: ",self.temperature)
        response = self.client.responses.create(
            model=self.model,
            input=prompt,
            temperature=self.temperature
        )
        return response.output_text.strip()


class GeminiAdapter(LLMStrategy):
    def __init__(self, api_key: Optional[str], model: str, temperature: float = 0.2):
        if not api_key:
            raise ValueError("GEMINI_API_KEY não encontrado.")
        self.api_key = api_key
        self.model = model
        self.temperature = temperature

        try:
            import google.generativeai as genai
        except ImportError as e:
            raise ImportError(
                "Pacote google-generativeai não está instalado. "
                "Instale com: pip install google-generativeai"
            ) from e

        self.genai = genai
        self.genai.configure(api_key=self.api_key)
        self.client = self.genai.GenerativeModel(self.model)

    def generate(self, prompt: str) -> str:
        response = self.client.generate_content(
            prompt,
            generation_config={
                "temperature": self.temperature
            }
        )
        text = getattr(response, "text", None)
        if not text:
            return ""
        return text.strip()


class LLMAdapterFactory:
    @staticmethod
    def create(provider: str) -> LLMStrategy:
        provider = provider.strip().lower()

        if provider == "openai":
            return OpenAIAdapter(
                api_key=os.getenv("OPENAI_API_KEY"),
                model=OPENAI_MODEL,
                temperature=LLM_TEMPERATURE
            )

        if provider == "gemini":
            return GeminiAdapter(
                api_key=os.getenv("GEMINI_API_KEY"),
                model=os.getenv("GEMINI_MODEL"),
                temperature=os.getenv("LLM_TEMPERATURE")
            )

        raise ValueError(
            f"Provider LLM não suportado: {provider}. "
            f"Use, por exemplo: 'openai' ou 'gemini'."
        )
    
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai").strip().lower()
llm_client = LLMAdapterFactory.create(LLM_PROVIDER)

def generate_with_llm(prompt_prefix: str, target: str, context: str, path):
    prompt = f"{prompt_prefix}\n\n{target}\n\n{context}"

    text = llm_client.generate(prompt)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(text, f, indent=2, ensure_ascii=False)
        
    return text

## Prompts modificados para cada tipo de contradição

In [40]:
prompt_temp ='''You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a temporal contradiction with the selected clause.

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

The following examples can guide you:

Good Examples:
Original: The report must be delivered within 10 days.
Contradiction: The report must be delivered within 30 days.

Original: Obligations begin 30 days after signature.
Contradiction: Obligations begin upon signature.

Bad Example:
Original: THI is granted an exclusive worldwide license to make, use, sell and import Licensed Products until the year of 2010.
Contradiction: THI may grant sublicenses to its Related Companies for the manufacture and distribution of Licensed Products after the 2008.
(This is a poor example since a party can simultaneously hold an exclusive license and grant sublicenses to its related companies; therefore, these situations are not mutually exclusive.)

If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT"
}

Contract:
'''

prompt_num ='''You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a numerical contradiction with the selected clause.

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

The following examples can guide you:

Good Examples:
Original: The total contract value shall be $10,000,000.
Contradiction: The total contract value shall be $5,000,000.

Original: The Supplier shall deliver 1,000 units per month.
Contradiction: The Supplier shall deliver 500 units per month.

Bad Example:
Original: The Buyer shall purchase a minimum of 1,000 units per quarter.
Contradiction: The Buyer may purchase up to 2,000 units per quarter.
(This is a poor example because both statements can be true at the same time; purchasing between 1,000 and 2,000 units satisfies both conditions, so they are not mutually exclusive.)

If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT"
}

Contract:
'''
prompt_auth ='''You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a authority contradiction with the selected clause, different source or issuer of a statement.

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

The following examples can guide you:

Good Examples:
Original: This policy is issued by the Compliance Office.
Contradiction: This policy is issued by the Strategy Unit.

Original: The report must be certified by an independent external auditor.
Contradiction: The report must be certified by the internal finance team.

Bad Example:
Original: This policy is issued by the Compliance Office.
Contradiction: This policy is reviewed by the Strategy Unit.
(This is a poor example because issuing and reviewing are different roles; both can occur simultaneously without conflict, so the statements are not mutually exclusive.)

If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT"
}

Contract:
'''
prompt_proc ='''You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a processual contradiction with the selected clause, conflicting procedures or operational routes.

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

The following examples can guide you:

Good Examples:
Original: Incident reports shall be filed using the internal ticketing system.
Contradiction: Incident reports shall be filed by contacting the operations team directly without using the ticketing system.

Original: All documents must be approved via the digital workflow tool before execution.
Contradiction: Documents may be executed without using the digital workflow tool or obtaining prior approval.

Bad Example:
Original: All reimbursement requests must be submitted through the HR portal.
Contradiction: All reimbursement requests must be approved by the finance department.
(This is a poor example because submission method and approval authority are different aspects; both statements can be true simultaneously, so there is no direct conflict.)

If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT"
}

Contract:
'''
prompt_pol ='''You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a policy reversal contradiction with the selected clause, one statement negates the other directly. .

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

The following examples can guide you:

Good Examples:
Original: Employees are allowed to access the system outside business hours.
Contradiction: Employees are prohibited from accessing the system outside business hours.

Original: The Contractor may subcontract services with prior approval.
Contradiction: The Contractor is not allowed to subcontract services under any circumstances.

Bad Example:
Original: THI is granted an exclusive worldwide license to make, use, sell and import Licensed Products.
Contradiction: THI may grant sublicenses to its Related Companies for the manufacture and distribution of Licensed Products.
(This is a poor example since a party can simultaneously hold an exclusive license and grant sublicenses to its related companies; therefore, these situations are not mutually exclusive.)

If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT"
}

Contract:
'''
prompt_spec ='''You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a specifity contradiction with the selected clause, one statement is more general or narrow than the other.

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

The following examples can guide you:

Good Examples:
Original: This policy applies globally to all Company operations.
Contradiction: This policy applies only to operations in the APAC region.

Original: The warranty covers all products delivered under this Agreement.
Contradiction: The warranty covers only defective products delivered under this Agreement.

Bad Example:
Original: This policy applies globally to all Company operations.
Contradiction: This policy applies to operations in Europe.
(This is a poor example because the second statement can be a subset of the first; both can be true simultaneously, so there is no contradiction.)

If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT"
}

Contract:  
'''

prompt_with_all = """
You are given a legal contract excerpt split into:
- TARGET_PARAGRAPH: the only paragraph that may receive a contradiction
- RELATED_PARAGRAPHS: contextual paragraphs that must NOT be edited, but may be used
  to make the overall set inconsistent.

Select ONE clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS that expresses
a concrete rule, fact, condition, or legal effect within the contract.

A valid clause may define:
- when something is valid or not (dates, deadlines, effective periods, conditions, prices)
- what happens under certain conditions
- ownership, rights, or responsibilities
- procedures, requirements, or constraints
- definitions of terms
- governing law or jurisdiction

Then generate ONE new sentence that creates a contradiction with the selected clause.
You may use different types of contradictions, including but not limited to:
- Temporal: Contradicts date or time of an event. Ex: “Starts Jan 15” vs. “Starts end of Q1”
- Numerical: Conflicting numbers, values, or percentages. Ex:“$12M surplus” vs. “$5M deficit”
- Authority: Different source or issuer of a statement. Ex: “Issued by Compliance Office” vs. “Issued by Strategy Unit”
- Process: Conflicting procedures or operational routes. Ex: “Submit via HR portal” vs. “Submit through admins”
- Policy Reversal: One statement negates the other directly. Ex: “Remote work mandatory” vs. “Remote work not permitted”
- Specificity: One statement is more general or narrow than the other. Ex: “Applies globally” vs. “Applies only to APAC”

The contradiction must:
1. contradict the selected clause from TARGET_PARAGRAPH or RELATED_PARAGRAPHS,
2. remain natural in a contract,
3. be fully coherent with the TARGET_PARAGRAPH (same subject and topic),
4. be inserted into TARGET_PARAGRAPH only.
5. use RELATED_PARAGRAPHS only as context if useful.

If the contradiction involves the target paragraph and a related paragraph, classify the scope as inter-paragraph. 
If the contradiction occurs within the same paragraph (self-contradiction), classify the scope as intra-paragraph.
The following examples can guide you:

Good Examples:
Original: This policy applies globally to all Company operations.
Contradiction: This policy applies only to operations in the APAC region.

Original: The Contractor may subcontract services with prior approval.
Contradiction: The Contractor is not allowed to subcontract services under any circumstances.

Original: The Supplier shall deliver 1,000 units per month.
Contradiction: The Supplier shall deliver 500 units per month.

Bad Examples:
Original: This policy applies globally to all Company operations.
Contradiction: This policy applies to operations in Europe.
(This is a poor example because the second statement can be a subset of the first; both can be true simultaneously, so there is no contradiction.)

Original: This policy is issued by the Compliance Office.
Contradiction: This policy is reviewed by the Strategy Unit.
(This is a poor example because issuing and reviewing are different roles; both can occur simultaneously without conflict, so the statements are not mutually exclusive.)


If you cannot find a clear contractual obligation in TARGET_PARAGRAPH, return [].
Do not explain anything.
Return JSON only.

Return a JSON list with exactly ONE object:
{
  "statement": "SELECTED_CLAUSE_FROM_TARGET_PARAGRAPH",
  "contradiction": "CONTRADICTORY_CLAUSE_TO_INSERT",
  "type_contradiction":"TYPE_OF_THE_CONTRADICTION_GENERATED"
  "scope_contradiction": "SCOPE_OF_THE_CONTRADICTION"
}

Contract:
"""

## Testando a geração de vários tipos de contradição

In [42]:
############### !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! ###############
# As informações de dias e preços estão censuradas como [...***...], para esse exemplo foi adicionada uma informação fictícia

context = "[RELATED_PARAGRAPH_1] (relation_type=semantic_similarity)\"Quality Agreement\" means one or more written agreements between the Parties, incorporating all relevant quality assurance and quality control obligations and aspects for the Parties with respect to the supply of Clinical Grade Products to Bellicum by Miltenyi under this Agreement.",
target = "[TARGET_PARAGRAPH]    (e) Quality Agreement. Within 15 days from the Effective Date (or such longer period as agreed by the Parties in writing, but in any event prior to the first delivery of Clinical Grade Product to Bellicum), the Parties shall enter into an agreement on mutually acceptable, commercially reasonable terms that details the quality assurance obligations of each Party relating to Clinical Grade Products (the \"Quality Agreement\"). In the event of a conflict between the terms of the Quality Agreement and the terms of this Agreement, the provisions of this Agreement shall govern; provided, however, that the Quality Agreement shall govern in respect of quality issues.",

text = generate_with_llm(prompt_num, target,context, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target,context, "all_types_contradiction1.json")
print(text)

Temperatura do modelo:  0.2
[
  {
    "statement": "Within 15 days from the Effective Date (or such longer period as agreed by the Parties in writing, but in any event prior to the first delivery of Clinical Grade Product to Bellicum), the Parties shall enter into an agreement on mutually acceptable, commercially reasonable terms that details the quality assurance obligations of each Party relating to Clinical Grade Products (the \"Quality Agreement\").",
    "contradiction": "The Parties shall enter into the Quality Agreement within 30 days from the Effective Date."
  }
]
Temperatura do modelo:  0.2
```json
[
  {
    "statement": "Within 15 days from the Effective Date (or such longer period as agreed by the Parties in writing, but in any event prior to the first delivery of Clinical Grade Product to Bellicum), the Parties shall enter into an agreement on mutually acceptable, commercially reasonable terms that details the quality assurance obligations of each Party relating to Clinica

In [43]:
target = "[TARGET_PARAGRAPH]    (b) In the event that Miltenyi becomes aware that it will not be able, or is likely not to be able, to produce all of Bellicum's forecast requirements of Miltenyi Products from its primary facility located in Bergisch Gladbach, Germany, Miltenyi shall determine, at its option and expense, to establish additional or alternative manufacturing and supply capability for the Miltenyi Products by qualifying and maintaining one or more back-up manufacturing facilities at the premises of Miltenyi and/or any of its Affiliates (each, a \"Secondary Location\"). Use of a Secondary Location must be notified to Bellicum in writing in accordance with the Change Notification processes set forth in Section 3.2. Miltenyi shall use its best efforts to provide to Bellicum with a commercially reasonable number of samples of the \"Secondary Location Miltenyi Products\" (meaning such Miltenyi Products that are produced at such Secondary Location) for evaluation by Bellicum as soon as each such Secondary Location Miltenyi Product becomes available during the post-noficiation period. In the event that Miltenyi decides to qualify a Secondary Location for the supply of Miltenyi Products hereunder, it shall provide reasonable prior written notice thereof (not less than"
context = """[RELATED_PARAGRAPH_1] (relation_type=semantic_similarity) six (6) months in advance) to Bellicum, including such details as Bellicum reasonably requires to assess the qualifications of such Secondary Location. Miltenyi shall have sole responsibility for all activities in connection with the setup and approval of the Secondary Location, including for establishing proof of product equivalence for Miltenyi Products produced at the Secondary Location, process and equipment validation and for filing all submissions or other correspondence with Miltenyi's applicable Regulatory Authorities in connection with the Secondary Location.
[RELATED_PARAGRAPH_2] (relation_type=semantic_similarity)(1) In the event of a Supply Failure (as defined below), Bellicum shall have the option to request Miltenyi to establish, as soon as reasonably feasible and at Miltenyi's sole cost and expense, a Secondary Location reasonably capable of making up the Supply Failure of the affected Miltenyi Product (the \"Affected Miltenyi Product\"), and if Miltenyi should either (i) notify Bellicum in writing that it is not willing and/or capable to establish a Secondary Location, or (ii) should not have established such Secondary Location and made up the Supply Failure within a reasonable period of time with regard to the Affected Miltenyi Product from receipt of Bellicum's written request therefore, then Bellicum shall, at Bellicum's sole cost and expense, have the right to select, qualify, and maintain an additional second source manufacturing facility as a back-up manufacturing facility for the Affected Miltenyi Products at the premises of a Third Party (the \"Second-Source Supplier\"). In the event that Bellicum elects to qualify a Second-Source Supplier for an Affected Miltenyi Product, it shall provide Miltenyi with prior written notice to Miltenyi including such details as Miltenyi reasonably requires to assess the qualifications of such Second-Source Supplier. Any such Second-Source Supplier shall be subject to the prior written consent of Miltenyi, which"""
text = generate_with_llm(prompt_proc, target,context, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target,context, "all_types_contradiction1.json")
print(text)

Temperatura do modelo:  0.2
[
  {
    "statement": "Use of a Secondary Location must be notified to Bellicum in writing in accordance with the Change Notification processes set forth in Section 3.2.",
    "contradiction": "Use of a Secondary Location may be implemented immediately without prior written notification to Bellicum or adherence to the Change Notification processes set forth in Section 3.2."
  }
]
Temperatura do modelo:  0.2
```json
[
  {
    "statement": "Miltenyi shall provide reasonable prior written notice thereof (not less than six (6) months in advance) to Bellicum, including such details as Bellicum reasonably requires to assess the qualifications of such Secondary Location.",
    "contradiction": "Miltenyi may establish a Secondary Location without providing any prior written notice to Bellicum.",
    "type_contradiction": "Process",
    "scope_contradiction": "inter-paragraph"
  }
]
```


In [44]:
target="20.3 Entire Agreement and Amendment. This Agreement (including all Exhibits attached hereto, which are incorporated herein by reference, and as amended from time to time in accordance with the provisions hereof) and any Quality Agreement(s) sets forth all of the covenants, promises, agreements, warranties, representations, conditions and understandings between the Parties hereto with respect to the subject matter hereof, and constitutes and contains the complete, final, and exclusive understanding and agreement of the Parties with respect to the subject matter hereof, and cancels, supersedes and terminates all prior agreements and understanding between the Parties with respect to the subject matter hereof. There are no covenants, promises, agreements, warranties, representations conditions or understandings, whether oral or written, between the Parties other than as set forth herein or in a Quality Agreement. No subsequent alteration, amendment, change or addition to this Agreement (including all Exhibits attached hereto) shall be binding upon the Parties hereto unless reduced to writing and signed by the respective authorized officers of the Parties.",
context = """[RELATED_PARAGRAPH_1] (relation_type=reference) 15.7 Survival. Other than obligations which have accrued and are outstanding as of the date of any expiration or termination of this Agreement, and except as otherwise expressly provided in this Agreement or the Quality Agreement or as otherwise mutually agreed by the Parties in writing, all rights granted and obligations undertaken by the Parties hereunder shall terminate immediately upon the termination or expiration of this Agreement, subject to Section 15.4 above and except for the following which shall survive according to their terms: Section 2.2 (Permitted Use); Section 2.7 (Subcontracting by Bellicum); Article 10 (Intellectual Property); Article 11 (Warranty); Article 12 (Limitation of Liability); Article 13 (Indemnification; Insurance); Article 14 (Confidentiality and Non-disclosure); Section 15.7 (Post-termination); Section 15.7 (Survival); Article 16 (Notices); Article 17 (Assignment); Article 19 (Dispute Resolution and Applicable Law); and Article 20 (Miscellaneous); and any and all rights and obligations of the Parties thereunder, as well as any other provision hereunder which by its nature is intended to survive expiration or termination of this Agreement.
[RELATED_PARAGRAPH_1] (relation_type=reference)  \"Agreement\" means this Supply Agreement, including Exhibits A, B, C, D, E, F and G attached hereto and incorporated herein, as amended from time to time in accordance with Section 20.3 hereof."""

text = generate_with_llm(prompt_auth, target,context, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target,context, "all_types_contradiction1.json")
print(text)

Temperatura do modelo:  0.2
```json
[
  {
    "statement": "No subsequent alteration, amendment, change or addition to this Agreement (including all Exhibits attached hereto) shall be binding upon the Parties hereto unless reduced to writing and signed by the respective authorized officers of the Parties.",
    "contradiction": "Any subsequent alteration, amendment, change or addition to this Agreement shall be binding upon the Parties hereto upon mutual oral agreement of the respective authorized officers."
  }
]
```
Temperatura do modelo:  0.2
[
  {
    "statement": "No subsequent alteration, amendment, change or addition to this Agreement (including all Exhibits attached hereto) shall be binding upon the Parties hereto unless reduced to writing and signed by the respective authorized officers of the Parties.",
    "contradiction": "Any verbal agreement or email confirmation between the Parties shall be binding and enforceable even if not reduced to writing or signed by authorized of

## Testando o prompt com todos os tipos em várias temperaturas

In [45]:
target_par1 ="[TARGET_PARAGRAPH]    2.5 Subcontracting by Miltenyi. Subject to the terms of the Quality Agreement, if applicable, Miltenyi may, at its sole discretion, upon reasonable prior written notice to Bellicum, elect to have the Miltenyi Products, or any one of them or any component thereof, manufactured by an Affiliate of Miltenyi, and further may subcontract the manufacturing of Miltenyi Product or any component thereof, to a Subcontractor; provided that (i) Miltenyi shall reasonably take into account Bellicum's written concerns regarding proposed Affiliate(s) or Subcontractor(s); and (ii) Miltenyi shall be solely and fully responsible for the performance of all delegated and subcontracted activities by its Affiliates and Subcontractor(s), including compliance with the terms of this Agreement and the Quality Agreement (as applicable), and in no event shall any such delegation or subcontract release Miltenyi from any of its obligations under this Agreement. Miltenyi's Subcontractors and Affiliates for the manufacture and/or supply of Miltenyi Products will be listed in the Quality Agreement"  
context_par1 ="""[RELATED_PARAGRAPHS]
[RELATED_PARAGRAPH_1] (relation_type=reference) \"Subcontractor\" means a Third Party to which, as applicable: (i) Miltenyi subcontracts the manufacture and/or supply of Miltenyi Products on behalf of Miltenyi and under Miltenyi's authority and responsibility in accordance with Section 2.5 and as further set forth in the Quality Agreement, if applicable; or (ii) Bellicum or its Licensees subcontracts the manufacture and/or supply of Bellicum Products on behalf of Bellicum or its Licensees and under Bellicum's or its Licensees' authority and responsibility in accordance with this Agreement and as described in the Bellicum Product specific Module attached hereto, as such Bellicum Product specific Module may be amended from time to time by written notification of Bellicum to Miltenyi to add or remove Subcontractor.
[RELATED_PARAGRAPH_2] (relation_type=semantic_similarity) (a) Miltenyi shall have sole responsibility for ensuring, and shall ensure, that Miltenyi's and its Affiliates' and Subcontractors' activities and performance in connection with the manufacture of Miltenyi Products and the supply of such Miltenyi Products to Bellicum under this Agreement are at all times in compliance with Applicable Laws. Without limiting the generality of the foregoing, it shall    
[RELATED_PARAGRAPH_3] (relation_type=semantic_similarity) 2.10 Liability for Non-Compliance. Notwithstanding anything to the contrary herein, Bellicum shall, in relation to Miltenyi, at all times and in all respects continue to remain fully and primarily responsible and liable to Miltenyi for the performance and the acts or omissions of its Affiliate, Subcontractor, and Licensee in connection with the subject matter of this Agreement, including the failure of an Affiliate, Subcontractor, or Licensee of Bellicum to comply with all of the limitations and obligations imposed on Bellicum hereunder. Notwithstanding anything to the contrary herein, Miltenyi shall, in relation to Bellicum, at all times and in all respects continue to remain fully and primarily responsible and liable to Bellicum for the performance and the acts or omissions of its Affiliates and Subcontractors in connection with the subject matter of this Agreement, including the failure of an Affiliate or Subcontractor of Miltenyi to comply with all of the limitations and obligations imposed on Miltenyi hereunder. For clarity, in no event shall any permitted delegation or subcontracting of any activities to be performed in connection with this Agreement release a Party from any of its limitations or obligations under this Agreement
[RELATED_PARAGRAPH_4] (relation_type=semantic_similarity) (c) In addition, Miltenyi may from time to time determine, in its sole discretion, to have one or more Miltenyi Products manufactured, assembled and/or supplied, in whole or in part, by a Subcontractor chosen by Miltenyi and reasonably acceptable to Bellicum. Miltenyi shall provide Bellicum with prior written notification of such Change in accordance with the applicable notification procedures as set forth in the Section Change Control and in the Quality Agreement, if applicable. Notwithstanding the foregoing, Miltenyi shall remain responsible for the fulfilment of its supply and other obligations hereunder with respect to any Miltenyi Product manufactured by Miltenyi's Subcontractor. Miltenyi shall be solely responsible for providing proof of product equivalence and for filing all submissions or other correspondence with the applicable governmental or regulatory authorities in connection with any decision to seek approval of a Third Party subcontractor site for the Miltenyi Products. Further, Miltenyi shall be solely responsible for all process and equipment validation required by the responsible Regulatory Authorities and the regulations thereunder and shall take all steps reasonably necessary to pass government inspection by such Regulatory Authorities"""

target_par2 ="[TARGET_PARAGRAPH]    (a) Product Specifications. Miltenyi shall manufacture or have manufactured the Miltenyi Products to meet the agreed Product Specifications, as then in effect, as published by Miltenyi from time to time, or as set forth in the Quality Agreement, as applicable."
context_par2 ="""[RELATED_PARAGRAPHS]
[RELATED_PARAGRAPH_1] (relation_type=semantic_similarity) (b) Agreed Standards. All Miltenyi Products shall be manufactured and quality controlled in compliance with and pursuant to: (i) the Agreed Standards, (ii) the requirements of the Quality Agreement, if applicable, and (iii) Applicable Laws
[RELATED_PARAGRAPH_2] (relation_type=semantic_similarity) (1) be manufactured, tested and Devilvered by Miltenyi in accordance with all applicable marketing approvals (if any), Agreed Standards, the terms of this Agreement and other Applicable Laws applicable at the place of manufacture to the manufacture, testing, and Delivery of Miltenyi Products by Miltenyi
[RELATED_PARAGRAPH_3] (relation_type=semantic_similarity) \"Product Specifications\" means the particulars as to composition, quality, safety, integrity, purity and other characteristics for a Miltenyi Product as published by Miltenyi from time to time, or as set forth in the applicable Quality Agreement entered into by the Parties in accordance with Section 3.2."""

In [46]:
text = generate_with_llm(prompt_with_all, target_par1,context_par1, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target_par2,context_par2, "all_types_contradiction2.json")
print(text)

Temperatura do modelo:  0.2
[
  {
    "statement": "Miltenyi shall be solely and fully responsible for the performance of all delegated and subcontracted activities by its Affiliates and Subcontractor(s), including compliance with the terms of this Agreement and the Quality Agreement (as applicable), and in no event shall any such delegation or subcontract release Miltenyi from any of its obligations under this Agreement.",
    "contradiction": "Miltenyi shall not be responsible for any failures or non-compliance arising from the actions or omissions of its Affiliates or Subcontractors once the manufacturing is subcontracted.",
    "type_contradiction": "Policy Reversal",
    "scope_contradiction": "intra-paragraph"
  }
]
Temperatura do modelo:  0.2
[
  {
    "statement": "Miltenyi shall manufacture or have manufactured the Miltenyi Products to meet the agreed Product Specifications, as then in effect, as published by Miltenyi from time to time, or as set forth in the Quality Agreement

In [29]:
text = generate_with_llm(prompt_with_all, target_par1,context_par1, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target_par2,context_par2, "all_types_contradiction2.json")
print(text)

Temperatura do modelo:  0.6
[
  {
    "statement": "Miltenyi shall be solely and fully responsible for the performance of all delegated and subcontracted activities by its Affiliates and Subcontractor(s), including compliance with the terms of this Agreement and the Quality Agreement (as applicable), and in no event shall any such delegation or subcontract release Miltenyi from any of its obligations under this Agreement.",
    "contradiction": "Miltenyi shall not be responsible for the performance of activities delegated to Subcontractors and Affiliates, and such delegation shall release Miltenyi from its obligations under this Agreement.",
    "type_contradiction": "Policy Reversal"
  }
]
Temperatura do modelo:  0.6
[
  {
    "statement": "Miltenyi shall manufacture or have manufactured the Miltenyi Products to meet the agreed Product Specifications, as then in effect, as published by Miltenyi from time to time, or as set forth in the Quality Agreement, as applicable.",
    "contradi

In [31]:
text = generate_with_llm(prompt_with_all, target_par1,context_par1, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target_par2,context_par2, "all_types_contradiction2.json")
print(text)

Temperatura do modelo:  0.85
```json
[
  {
    "statement": "Miltenyi shall be solely and fully responsible for the performance of all delegated and subcontracted activities by its Affiliates and Subcontractor(s), including compliance with the terms of this Agreement and the Quality Agreement (as applicable), and in no event shall any such delegation or subcontract release Miltenyi from any of its obligations under this Agreement.",
    "contradiction": "Miltenyi shall not be responsible for the performance of any activities delegated to its Affiliates or Subcontractors, and such delegation shall release Miltenyi from all obligations under this Agreement.",
    "type_contradiction": "Policy Reversal"
  }
]
```
Temperatura do modelo:  0.85
[
  {
    "statement": "Miltenyi shall manufacture or have manufactured the Miltenyi Products to meet the agreed Product Specifications, as then in effect, as published by Miltenyi from time to time, or as set forth in the Quality Agreement, as applic

In [ ]:
text = generate_with_llm(prompt_with_all, target_par1,context_par1, "all_types_contradiction1.json")
print(text)

text = generate_with_llm(prompt_with_all, target_par2,context_par2, "all_types_contradiction2.json")
print(text)

Temperatura do modelo:  0.95
```json
[
  {
    "statement": "Miltenyi shall be solely and fully responsible for the performance of all delegated and subcontracted activities by its Affiliates and Subcontractor(s), including compliance with the terms of this Agreement and the Quality Agreement (as applicable), and in no event shall any such delegation or subcontract release Miltenyi from any of its obligations under this Agreement.",
    "contradiction": "Miltenyi shall not be liable for any outsourced activities performed by its Affiliates or Subcontractors, and such delegation shall release Miltenyi from obligations related to those activities under this Agreement.",
    "type_contradiction": "Policy Reversal"
  }
]
```
Temperatura do modelo:  0.95
[
  {
    "statement": "Miltenyi shall manufacture or have manufactured the Miltenyi Products to meet the agreed Product Specifications, as then in effect, as published by Miltenyi from time to time, or as set forth in the Quality Agreement